In [27]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_csv("/home/shavit/asf/bench/results/presolver_benchmark.csv")
df.drop(columns=["schedule", "maximize"], inplace=True)
df

,scenario,fold,presolver,presolver_budget,n_algorithms_total,n_train_instances,n_test_instances,train_n_solved,train_solve_rate,train_total_runtime,...,test_solve_rate,test_total_runtime,test_avg_runtime,vbs_score,sbs_score,wall_time_seconds,cpu_time_seconds,schedule_length,schedule_cost,scenario_budget
0,TSP-LION2015,1,Greedy_cutoff5,10.0,4,2795,311,747,0.267263,7.375096e+07,...,0.318328,7634440.616,24548.040566,5493.039,16555.353,0.000363,0.000352,2,10.000,3600.0
1,TSP-LION2015,1,Greedy_cutoff10,10.0,4,2795,311,1167,0.417531,5.862974e+07,...,0.482315,5798289.951,18644.019135,5493.039,16555.353,0.000274,0.000272,1,10.000,3600.0
2,TSP-LION2015,1,Greedy_cutoff5_max5,10.0,4,2795,311,747,0.267263,7.375096e+07,...,0.318328,7634440.616,24548.040566,5493.039,16555.353,0.000289,0.000285,2,10.000,3600.0
3,TSP-LION2015,1,Submodular_default,10.0,4,2795,311,747,0.267263,7.375096e+07,...,0.318328,7634440.616,24548.040566,5493.039,16555.353,0.000689,0.000683,2,10.000,3600.0
4,TSP-LION2015,1,Submodular_fine,10.0,4,2795,311,747,0.267263,7.375096e+07,...,0.318328,7634440.616,24548.040566,5493.039,16555.353,0.000923,0.000918,2,10.000,3600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,BNSL-2016,4,ASAPv2_medium,100.0,8,1061,118,444,0.418473,4.443120e+07,...,0.338983,5616873.337,47600.621500,26399.160,135351.730,0.087523,0.087018,3,9.999,7200.0
5396,BNSL-2016,4,ASAPv2_large,100.0,8,1061,118,459,0.432611,4.335123e+07,...,0.372881,5328861.967,45159.847178,26399.160,135351.730,0.085808,0.084936,3,10.000,7200.0
5397,BNSL-2016,4,Static3S_default,100.0,8,1061,118,739,0.696513,2.323218e+07,...,0.720339,2381622.280,20183.239661,26399.160,135351.730,1.809620,0.535432,3,100.000,7200.0
5398,BNSL-2016,4,Static3S_fine,100.0,8,1061,118,745,0.702168,2.279965e+07,...,0.720339,2381506.340,20182.257119,26399.160,135351.730,3.840316,1.307642,3,100.000,7200.0


presolver_budget,10.0,20.0,30.0,50.0,100.0
presolver,,,,,
ASAPv2_large,31.722222,31.755556,31.788889,31.788889,42.144444
ASAPv2_medium,30.755556,30.911111,30.911111,30.944444,41.888889
ASAPv2_small,29.788889,29.833333,29.844444,29.844444,42.222222
Aspeed_default,30.366667,37.766667,39.522222,43.811111,47.633333
Greedy_cutoff10,41.344444,47.944444,49.577778,49.577778,49.577778
Greedy_cutoff5,36.311111,37.522222,37.522222,37.522222,37.522222
Greedy_cutoff5_max5,36.311111,37.877778,37.966667,37.966667,37.966667
Static3S_default,42.744444,52.455556,57.466667,64.722222,72.144444
Static3S_fine,42.866667,52.655556,57.677778,65.100000,72.466667


In [26]:
df["presolver"].unique()

array(['Greedy_cutoff5', 'Greedy_cutoff10', 'Greedy_cutoff5_max5',
       'Submodular_default', 'Submodular_fine', 'Submodular_coarse',
       'ASAPv2_small', 'ASAPv2_medium', 'ASAPv2_large',
       'Static3S_default', 'Static3S_fine', 'Aspeed_default'],
      dtype=object)

In [38]:
display_names = {
    "Greedy_cutoff5": "Greedy (cutoff=5s)",
    "Greedy_cutoff10": "Greedy (cutoff=10s)",
    "Greedy_cutoff5_max5": "Greedy (cutoff=5s, max 5 algs)",
    "Submodular_default": "Submodular",
    "Submodular_fine": "Submodular (fine)",
    "Submodular_coarse": "Submodular (coarse)",
    "Static3S_default"  : "Static3S",
    "Static3S_fine"     : "Static3S (fine)",
    "Aspeed_default"   : "Aspeed",
    "ASAPv2_small": "ASAPv2 (small)",
    "ASAPv2_medium": "ASAPv2 (medium)",
    "ASAPv2_large": "ASAPv2 (large)",
}

pivot = df.groupby(["presolver", "presolver_budget"]).mean(numeric_only=True)["test_solve_rate"].unstack()

pivot = pivot * 100  # Convert to percentage

# Rename index for display
pivot_display = pivot.rename(index=display_names)

# Filter AFTER renaming
pivot_display = pivot_display[pivot_display.index != "ASAPv2 (small)"]
pivot_display = pivot_display[pivot_display.index != "ASAPv2 (medium)"]
pivot_display = pivot_display[pivot_display.index != "Greedy (cutoff=5s)"]
pivot_display = pivot_display[pivot_display.index != "Greedy (cutoff=5s, max 5 algs)"]
pivot_display = pivot_display[pivot_display.index != "Submodular (fine)"]
pivot_display = pivot_display[pivot_display.index != "Submodular (coarse)"]
pivot_display = pivot_display[pivot_display.index != "Static3S (fine)"]

import plotly.graph_objects as go

# Create text array with empty strings for NaN values
text_array = [
    [f"{v:.1f}" if not np.isnan(v) else "" for v in row] for row in pivot_display.values
]

# Target ~300x300 for the plot area, add space for labels/ticks/titles
plot_size = 225
left_margin = 140
bottom_margin = 60

fig = go.Figure(
    data=go.Heatmap(
        z=pivot_display.values,
        x=[str(c) for c in pivot_display.columns],
        y=pivot_display.index.tolist(),
        text=text_array,
        texttemplate="%{text}",
        textfont=dict(size=8),  # Small font to fit in cells
        colorscale="Plasma",
        zmin=np.nanmin(pivot_display.values),
        zmax=np.nanmax(pivot_display.values),
        hoverongaps=False,
        showscale=False,
        xgap=0,
        ygap=0,
    )
)

fig.update_layout(
    xaxis_title="Presolving Timeout (seconds)",
    yaxis_title=None,
    width=plot_size + left_margin,
    height=plot_size + bottom_margin,
    font=dict(size=8),
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(len(pivot_display.index))),
        ticktext=pivot_display.index.tolist(),
        autorange="reversed",
        tickangle=30,
        tickfont=dict(size=8),
    ),
    xaxis=dict(
        tickmode="array",
        tickvals=[str(c) for c in pivot_display.columns],
        ticktext=[str(c) for c in pivot_display.columns],
        tickfont=dict(size=8),
        title_font=dict(size=8),
    ),
    margin=dict(l=5, r=5, t=5, b=5),
    plot_bgcolor="white",
)
fig.update_traces(showlegend=False)
fig.write_image("/home/shavit/asf/paper/ol_v1/figures/presolving_heatmap.pdf")

In [39]:
display_names = {
    "Greedy_cutoff5": "Greedy (cutoff=5s)",
    "Greedy_cutoff10": "Greedy (cutoff=10s)",
    "Greedy_cutoff5_max5": "Greedy (cutoff=5s, max 5 algs)",
    "Submodular_default": "Submodular",
    "Submodular_fine": "Submodular (fine)",
    "Submodular_coarse": "Submodular (coarse)",
    "Static3S_default"  : "Static3S",
    "Static3S_fine"     : "Static3S (fine)",
    "Aspeed_default"   : "Aspeed",
    "ASAPv2_small": "ASAPv2 (small)",
    "ASAPv2_medium": "ASAPv2 (medium)",
    "ASAPv2_large": "ASAPv2 (large)",
}

pivot = df.groupby(["presolver", "presolver_budget"]).mean(numeric_only=True)["cpu_time_seconds"].unstack()

pivot = pivot * 1  # Convert to percentage

# Rename index for display
pivot_display = pivot.rename(index=display_names)

# Filter AFTER renaming
pivot_display = pivot_display[pivot_display.index != "ASAPv2 (small)"]
pivot_display = pivot_display[pivot_display.index != "ASAPv2 (medium)"]
pivot_display = pivot_display[pivot_display.index != "Greedy (cutoff=5s)"]
pivot_display = pivot_display[pivot_display.index != "Greedy (cutoff=5s, max 5 algs)"]
pivot_display = pivot_display[pivot_display.index != "Submodular (fine)"]
pivot_display = pivot_display[pivot_display.index != "Submodular (coarse)"]
pivot_display = pivot_display[pivot_display.index != "Static3S (fine)"]

import plotly.graph_objects as go

# Create text array with empty strings for NaN values
text_array = [
    [f"{v:.1f}" if not np.isnan(v) else "" for v in row] for row in pivot_display.values
]

# Target ~300x300 for the plot area, add space for labels/ticks/titles
plot_size = 225
left_margin = 140
bottom_margin = 60

fig = go.Figure(
    data=go.Heatmap(
        z=pivot_display.values,
        x=[str(c) for c in pivot_display.columns],
        y=pivot_display.index.tolist(),
        text=text_array,
        texttemplate="%{text}",
        textfont=dict(size=8),  # Small font to fit in cells
        colorscale="Plasma",
        zmin=np.nanmin(pivot_display.values),
        zmax=np.nanmax(pivot_display.values),
        hoverongaps=False,
        showscale=False,
        xgap=0,
        ygap=0,
    )
)

fig.update_layout(
    xaxis_title="Presolving Timeout (seconds)",
    yaxis_title=None,
    width=plot_size + left_margin,
    height=plot_size + bottom_margin,
    font=dict(size=8),
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(len(pivot_display.index))),
        ticktext=pivot_display.index.tolist(),
        autorange="reversed",
        tickangle=30,
        tickfont=dict(size=8),
    ),
    xaxis=dict(
        tickmode="array",
        tickvals=[str(c) for c in pivot_display.columns],
        ticktext=[str(c) for c in pivot_display.columns],
        tickfont=dict(size=8),
        title_font=dict(size=8),
    ),
    margin=dict(l=5, r=5, t=5, b=5),
    plot_bgcolor="white",
)
fig.update_traces(showlegend=False)
fig.write_image("/home/shavit/asf/paper/ol_v1/figures/presolving_heatmap_cpu_time.pdf")